In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# Giải nén tập dữ liệu 
%cd /content
!unzip -q /content/drive/MyDrive/EdgeCard_System/dataset_det.zip

In [ ]:
# Cài đặt Python 3.10 và hệ thống quản lý thư viện
%cd /content
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils -y
!wget https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py

# Cài đặt PaddleOCR cho DBNet và PP-OCRv4
!python3.10 -m pip install paddlepaddle-gpu
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd /content/PaddleOCR
!python3.10 -m pip install -r requirements.txt

In [ ]:
%cd /content/PaddleOCR
!mkdir -p pretrain_models
!wget -nc -P pretrain_models/ https://paddleocr.bj.bcebos.com/pretrained/MobileNetV3_large_x0_5_pretrained.pdparams

import os

os.makedirs('configs/det', exist_ok=True)

yaml_content = """Global:
  use_gpu: true
  use_xpu: false
  use_mlu: false
  epoch_num: 50
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: /content/drive/MyDrive/EdgeCard_System/stage_2/dbnet
  save_epoch_step: 10
  eval_batch_step: [0, 60]
  cal_metric_during_train: False
  pretrained_model: ./pretrain_models/MobileNetV3_large_x0_5_pretrained
  checkpoints:
  save_inference_dir:
  use_visualdl: False
  infer_img:
  save_res_path: /content/drive/MyDrive/EdgeCard_System/stage_2/dbnet/predicts_db.txt

Architecture:
  model_type: det
  algorithm: DB
  Transform:
  Backbone:
    name: MobileNetV3
    scale: 0.5
    model_name: large
  Neck:
    name: DBFPN
    out_channels: 256
  Head:
    name: DBHead
    k: 50

Loss:
  name: DBLoss
  balance_loss: true
  main_loss_type: DiceLoss
  alpha: 5
  beta: 10
  ohem_ratio: 3

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.001
    warmup_epoch: 2
  regularizer:
    name: 'L2'
    factor: 0

PostProcess:
  name: DBPostProcess
  thresh: 0.3
  box_thresh: 0.6
  max_candidates: 1000
  unclip_ratio: 1.5

Metric:
  name: DetMetric
  main_indicator: hmean

Train:
  dataset:
    name: SimpleDataSet
    data_dir: /content/dataset_det/
    label_file_list:
      - /content/dataset_det/train_label.txt
    ratio_list: [1.0]
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: False
      - DetLabelEncode:
      - IaaAugment:
          augmenter_args:
            - { 'type': Affine, 'args': { 'rotate': [-10, 10] } }
      - EastRandomCropData:
          size: [640, 640]
          max_tries: 50
          keep_ratio: true
      - MakeBorderMap:
          shrink_ratio: 0.4
          thresh_min: 0.3
          thresh_max: 0.7
      - MakeShrinkMap:
          shrink_ratio: 0.4
          min_text_size: 8
      - NormalizeImage:
          scale: 1./255.
          mean: [0.485, 0.456, 0.406]
          std: [0.229, 0.224, 0.225]
          order: 'hwc'
      - ToCHWImage:
      - KeepKeys:
          keep_keys: ['image', 'threshold_map', 'threshold_mask', 'shrink_map', 'shrink_mask']
  loader:
    shuffle: True
    drop_last: False
    batch_size_per_card: 16
    num_workers: 2
    use_shared_memory: False

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: /content/dataset_det/
    label_file_list:
      - /content/dataset_det/valid_label.txt
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: False
      - DetLabelEncode:
      - DetResizeForTest:
          image_shape: [736, 1280]
      - NormalizeImage:
          scale: 1./255.
          mean: [0.485, 0.456, 0.406]
          std: [0.229, 0.224, 0.225]
          order: 'hwc'
      - ToCHWImage:
      - KeepKeys:
          keep_keys: ['image', 'shape', 'polys', 'ignore_tags']
  loader:
    shuffle: False
    drop_last: False
    batch_size_per_card: 1
    num_workers: 2
    use_shared_memory: False
"""

# Ghi ra file
with open('configs/det/custom_db_mv3_v2.yml', 'w', encoding='utf-8') as f:
    f.write(yaml_content)

print("Đã tạo file cấu hình chuẩn custom_db_mv3_v2.yml thành công!")

# Kích hoạt huấn luyện
# !python3.10 tools/train.py -c configs/det/custom_db_mv3_v2.yml

In [ ]:
%cd /content/PaddleOCR

# Đánh giá DBNet
!python3.10 tools/eval.py \
    -c configs/det/custom_db_mv3_v2.yml \
    -o Global.checkpoints=/content/drive/MyDrive/EdgeCard_System/stage_2/dbnet/best_accuracy \
       Eval.dataset.label_file_list=["/content/dataset_det/test_label.txt"]